In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Wits_Honours_Project
!git pull

/content/drive/MyDrive/Wits_Honours_Project
Already up to date.


In [ ]:
!ls pipeline/llm
!ls pipeline/configs/llm

01_create_temporal_splits.py	     07_evaluate_recommendations.py
02_build_sft_datasets.py	     08_compare_results.py
03_validate_sft_datasets.py	     common.py
04_train_qlora.py		     constrained_decoding.py
05_select_best_checkpoint.py	     __init__.py
06_generate_test_recommendations.py  __pycache__
llama_fashion_qlora.yaml      qwen_fashion_qlora.yaml
llama_video_games_qlora.yaml  qwen_video_games_qlora.yaml


In [ ]:
!pip install -q -r requirements_qlora.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 60.4 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

NVIDIA A100-SXM4-80GB
BF16 supported: True


In [ ]:
!python pipeline/llm/04_train_qlora.py \
    pipeline/configs/llm/qwen_fashion_qlora.yaml

QLORA SEQUENTIAL RECOMMENDER TRAINING
Experiment:       qwen_fashion_qlora
Dataset:          Fashion
Base model:       Qwen/Qwen2.5-3B-Instruct
GPU:              NVIDIA A100-SXM4-80GB
GPU memory:       79.25 GiB
Compute dtype:    torch.bfloat16
Training file:    /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/sft/train.jsonl
Validation file:  /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/sft/validation.jsonl
Output directory: /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/qwen

Loading SFT datasets...
Generating train split: 12036 examples [00:00, 82335.68 examples/s]
Generating validation split: 3135 examples [00:01, 2990.09 examples/s]
Preparing training conversations (num_proc=2): 100% 12036/12036 [00:00<00:00, 13223.52 examples/s]
Preparing validation conversations (num_proc=2): 100% 3135/3135 [00:00<00:00, 8234.62 examples/s]
Training examples:   12,036
Validation examples: 3,135

Loadi

In [ ]:
!python pipeline/llm/05_select_best_checkpoint.py \
    pipeline/configs/llm/qwen_fashion_qlora.yaml

QLORA CHECKPOINT SELECTION
Model:              Qwen/Qwen2.5-3B-Instruct
Checkpoints:        3
Validation users:   3,135
Catalogue items:    12,493
Top K:              10
Beam width:         20
Batch size:         4
GPU:                NVIDIA A100-SXM4-80GB

config.json: 100% 661/661 [00:00<00:00, 3.53MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 4.47MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 14.2MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 12.8MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 20.3MB/s]
Loading base model in 4-bit...
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 78.8MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  11% 658M/6.17G [00:01<00:08, 649MB/s,  496MB/s  ]
Reconstructing (incomplete total...):  63% 3.90G/6.17G [00:06<00:04, 55

In [ ]:
!python pipeline/llm/06_generate_test_recommendations.py \
    pipeline/configs/llm/qwen_fashion_qlora.yaml

GENERATING TEST RECOMMENDATIONS
Dataset:             dataset
Base model:          Qwen/Qwen2.5-3B-Instruct
Selected checkpoint: /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/qwen/checkpoint-375
Test file:           /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/test.jsonl
Catalogue file:      /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/item_catalogue.json
Predictions file:    /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/qwen/test_predictions/test_predictions.jsonl
Top-K:               10
Beams:               20
Batch size:          4
Exclude history:     True

Loading test records...
Test records: 3,135
Loading metadata...
Metadata records: 12,493
Loading item catalogue...
Catalogue items: 12,493
Checking test records...
Already completed: 0
Remaining records: 3,135

Loading tokenizer and selected QLoRA adapter...
[transformers] `to

In [ ]:
!python pipeline/llm/07_evaluate_recommendations.py \
    pipeline/configs/llm/qwen_fashion_qlora.yaml

EVALUATING TEST RECOMMENDATIONS
Dataset:          Fashion
Model:            Qwen/Qwen2.5-3B-Instruct
Test split:       /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/test.jsonl
Predictions:      /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/qwen/test_predictions/test_predictions.jsonl
Catalogue:        /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/item_catalogue.json
Cutoffs:          [1, 5, 10]
Metrics output:   /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/qwen/test_predictions/test_metrics.json
Per-user output:  /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/qwen/test_predictions/per_user_metrics.jsonl

Loading authoritative test split...
Test users: 3,135
Loading predictions...
Prediction records: 3,135
Loading catalogue...
Catalogue items: 12,493
Evaluating users: 100% 3135/3135 [00:00<00:00

In [ ]:
!python pipeline/llm/04_train_qlora.py \
    pipeline/configs/llm/llama_fashion_qlora.yaml

QLORA SEQUENTIAL RECOMMENDER TRAINING
Experiment:       llama_fashion_qlora
Dataset:          Fashion
Base model:       meta-llama/Llama-3.2-3B-Instruct
GPU:              NVIDIA A100-SXM4-80GB
GPU memory:       79.25 GiB
Compute dtype:    torch.bfloat16
Training file:    /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/sft/train.jsonl
Validation file:  /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/sft/validation.jsonl
Output directory: /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/llama

Loading SFT datasets...
Generating train split: 12036 examples [00:00, 81096.88 examples/s]
Generating validation split: 3135 examples [00:00, 5636.24 examples/s]
Preparing training conversations (num_proc=2): 100% 12036/12036 [00:00<00:00, 13086.40 examples/s]
Preparing validation conversations (num_proc=2): 100% 3135/3135 [00:00<00:00, 8536.58 examples/s] 
Training examples:   12,036
Validation examples: 3

In [ ]:
!python pipeline/llm/05_select_best_checkpoint.py \
    pipeline/configs/llm/llama_fashion_qlora.yaml

QLORA CHECKPOINT SELECTION
Model:              meta-llama/Llama-3.2-3B-Instruct
Checkpoints:        3
Validation users:   3,135
Catalogue items:    12,493
Top K:              10
Beam width:         20
Batch size:         4
GPU:                NVIDIA A100-SXM4-80GB

config.json: 100% 878/878 [00:00<00:00, 3.39MB/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 11.3MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 22.8MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.41MB/s]
Loading base model in 4-bit...
model.safetensors.index.json: 100% 20.9k/20.9k [00:00<00:00, 38.5MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.97G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  13% 848M/6.43G [00:02<00:09, 594MB/s,  408MB/s  ]
Reconstructing (incomplete total...):  23% 1.51G/6.43G [00:03<00:07, 633MB/s,  616MB/s  ]
Reconstructing (i

In [ ]:
!python pipeline/llm/06_generate_test_recommendations.py \
    pipeline/configs/llm/llama_fashion_qlora.yaml

GENERATING TEST RECOMMENDATIONS
Dataset:             Fashion
Base model:          meta-llama/Llama-3.2-3B-Instruct
Selected checkpoint: /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/llama/checkpoint-1131
Test file:           /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/test.jsonl
Catalogue file:      /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/item_catalogue.json
Predictions file:    /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/llama/test_predictions/test_predictions.jsonl
Top-K:               10
Beams:               20
Batch size:          4
Exclude history:     True

Loading test records...
Test records: 3,135
Loading metadata...
Metadata records: 12,493
Loading item catalogue...
Catalogue items: 12,493
Checking test records...
Already completed: 0
Remaining records: 3,135

Loading tokenizer and selected QLoRA adapter...
config.

In [ ]:
!python pipeline/llm/07_evaluate_recommendations.py \
    pipeline/configs/llm/llama_fashion_qlora.yaml

EVALUATING TEST RECOMMENDATIONS
Dataset:          Fashion
Model:            meta-llama/Llama-3.2-3B-Instruct
Test split:       /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/test.jsonl
Predictions:      /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/llama/test_predictions/test_predictions.jsonl
Catalogue:        /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/splits/item_catalogue.json
Cutoffs:          [1, 5, 10]
Metrics output:   /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/llama/test_predictions/test_metrics.json
Per-user output:  /content/drive/MyDrive/Wits_Honours_Project/datasets/Fashion/processed/llm/checkpoints/llama/test_predictions/per_user_metrics.jsonl

Loading authoritative test split...
Test users: 3,135
Loading predictions...
Prediction records: 3,135
Loading catalogue...
Catalogue items: 12,493
Evaluating users: 100% 3135/3135 [